# When does a minute end?
**Code follow-up to the paper activity**

Did the paper sheet in class? Here's the same stream in code. Run it to check your watermark trace and see how a real system would do it.

Run the supplied Python/pandas cells with Shift+Enter; no installs or data downloads.
Choose **File → Save a copy in Drive** first. Write predictions in the text cells
before running the next code cell. All required cells run top to bottom unchanged.
The last Spark cell is optional and skipped by default.

We count **observations per minute**, not passengers or distinct buses.
All data is fictional, on one date in UTC. A dozen observations models ongoing
input; this finite replay is not itself evidence of big data.

The terms are the same as the dashboard model:
- **Event time:** when the event happened.
- **Processing time:** when the processing system observes it. Here `arrival_time`
  represents processing time; Colab's current clock does not affect the replay.
- **Time window:** a finite group using a chosen clock. Include its start, exclude its end.
- **State:** information retained between arrivals. **Finalize:** stop updating a minute.
- **Watermark:** advancing estimate of event-time progress/completeness, not proof.
- **Allowed delay:** chosen tolerance behind the largest event time seen.
- **Late log:** observations for retired minutes. Retired minutes end at/before the watermark.

A **simulator** runs a simplified model. Only the supplied arrivals advance this one.

## 1. Inspect the observations
One row is one observation. Equal arrival times use observation ID order.
Run the data cell. Find I and J; compare their two times.

In [ ]:
import pandas as pd
from copy import deepcopy

# Authored, fictional observations. One row is an observation, not a distinct bus.
rows = [
    ("F", "bus-1", "12:00:10", "12:00:12"),
    ("G", "bus-2", "12:01:20", "12:01:22"),
    ("H", "bus-1", "12:02:20", "12:02:22"),
    ("I", "bus-2", "12:01:10", "12:02:25"),
    ("J", "bus-1", "12:00:50", "12:02:25"),
    ("K", "bus-2", "12:03:00", "12:03:02"),
    ("L", "bus-1", "12:02:10", "12:03:05"),
    ("M", "bus-2", "12:01:55", "12:03:06"),
    ("N", "bus-1", "12:04:00", "12:04:02"),
    ("O", "bus-2", "12:02:55", "12:04:05"),
    ("P", "bus-1", "12:03:40", "12:04:06"),
    ("Q", "bus-2", "12:04:20", "12:04:22"),
]
observations = pd.DataFrame(rows, columns=["observation_id", "bus_id", "event_time", "arrival_time"])
for column in ("event_time", "arrival_time"):
    observations[column] = pd.to_datetime("2026-11-04 " + observations[column], utc=True)
observations = observations.sort_values(["arrival_time", "observation_id"]).reset_index(drop=True)
print(observations.to_string(index=False))

## 2. Choose the clock, then count
**Predict first:** will the 12:02 event-time and processing-time buckets contain the
same IDs? If their counts agree, what would that establish?

**Prediction:** _Write here._

In [ ]:
# First group by processing time, then by event time. Count observations.
processing_minutes = observations.arrival_time.dt.floor("min")
event_minutes = observations.event_time.dt.floor("min")
processing_counts = observations.groupby(processing_minutes).size().rename("processing_count")
event_counts = observations.groupby(event_minutes).size().rename("event_count")
count_comparison = pd.concat([processing_counts, event_counts], axis=1).fillna(0).astype(int)
print(count_comparison.to_string())

# Equal counts need not mean equal membership. Inspect one minute.
minute_to_inspect = pd.Timestamp("2026-11-04 12:02:00", tz="UTC")
print("Processing-time IDs:", observations.loc[processing_minutes == minute_to_inspect, "observation_id"].tolist())
print("Event-time IDs:", observations.loc[event_minutes == minute_to_inspect, "observation_id"].tolist())

**Explain:** use the displayed IDs to explain what count agreement does and does not
establish. Which clock would you use to report when observations happened?
Write K's event-time minute as [start, end).

**Your explanation:** _Write here._

## 3. Watch a minute finalize
The supplied code applies the same classroom policy:
1. Flag event time < **previous** watermark.
2. If minute end ≤ previous watermark, late-log the record; otherwise count it.
3. Update largest event time; watermark = largest event time minus allowed delay.
4. Finalize/remove active minutes ending at/before the new watermark.

“Behind watermark” and “minute retired” are separate tests. This policy is our
simplified model, not a promise about every engine. Run the supplied definition
cell once, then the play cells. You do not need to edit the implementation.

In [ ]:
def clock(value):
    return "none" if value is None else value.strftime("%H:%M:%S")

def minute_counts(values):
    return {k.strftime("%H:%M"): v for k, v in sorted(values.items())}

class ObservationStream:
    """The supplied classroom policy; not a full Spark implementation."""
    def __init__(self, data, allowed_delay_seconds):
        if not isinstance(allowed_delay_seconds, (int, float)) or allowed_delay_seconds < 0:
            raise ValueError("Allowed delay must be a nonnegative number of seconds")
        self.data = data.sort_values(["arrival_time", "observation_id"]).reset_index(drop=True).copy()
        self.allowed_delay_seconds = allowed_delay_seconds
        self.position = 0
        self.maximum_event_time = None
        self.watermark = None
        self.active = {}
        self.finalized = {}
        self.late_log = []
        self.history = []

    def snapshot(self):
        return deepcopy({"watermark": clock(self.watermark),
                         "active": minute_counts(self.active),
                         "finalized": minute_counts(self.finalized),
                         "late_log": self.late_log})

    def next_observation(self):
        if self.position == len(self.data):
            return None  # No final flush: the ongoing input has no known end.
        row = self.data.iloc[self.position]
        event_time = row.event_time
        minute = event_time.floor("min")
        minute_end = minute + pd.Timedelta(minutes=1)
        previous_watermark = self.watermark
        behind = previous_watermark is not None and event_time < previous_watermark
        retired = previous_watermark is not None and minute_end <= previous_watermark
        if retired:
            self.late_log.append(row.observation_id)
            route = "late log"
        else:
            self.active[minute] = self.active.get(minute, 0) + 1
            route = "count"
        self.maximum_event_time = (event_time if self.maximum_event_time is None
                                   else max(self.maximum_event_time, event_time))
        self.watermark = self.maximum_event_time - pd.Timedelta(seconds=self.allowed_delay_seconds)
        for start in list(self.active):
            if start + pd.Timedelta(minutes=1) <= self.watermark:
                self.finalized[start] = self.active.pop(start)
        self.position += 1
        step = {"id": row.observation_id, "event_time": clock(event_time),
                "arrival_time": clock(row.arrival_time), "minute_end": clock(minute_end),
                "previous_watermark": clock(previous_watermark), "behind_watermark": behind,
                "route": route, **self.snapshot()}
        self.history.append(deepcopy(step))
        return step

# Run this supplied play function repeatedly: n observations are released per call.
def play(stream, n=1, show=True):
    released = []
    for _ in range(n):
        step = stream.next_observation()
        if step is None:
            break
        released.append(step)
        if show:
            print("\n", step["id"], "event", step["event_time"], "arrival", step["arrival_time"],
                  "previous watermark", step["previous_watermark"])
            print("Behind watermark:", step["behind_watermark"], "minute end:", step["minute_end"],
                  "route:", step["route"], "new watermark:", step["watermark"])
            print("Active:", step["active"], "Finalized:", step["finalized"], "Late log:", step["late_log"])
    if not released and show:
        print("No supplied observations remain. Watermark and state stay unchanged.")
    return released

def replay(delay):
    fresh = ObservationStream(observations, delay)
    play(fresh, len(observations), show=False)
    return fresh

In [ ]:
# Reset all state; release F, G, H in arrival order.
stream = ObservationStream(observations, allowed_delay_seconds=30)
_ = play(stream, n=3)

**Predict before playing I and J:** compare each event time and minute end with the
previous watermark. Do they take the same route? Explain the two tests separately.

**Your prediction:** _Write here._

In [ ]:
# Play releases the next two arrivals; re-running it releases later observations.
# To restart, rerun the reset cell above, not just this cell.
_ = play(stream, n=2)

**Your explanation after observing I and J:** _Write here._

Release the remaining arrivals. Each release prints the previous watermark, flag,
route, new watermark and retained/finalized counts. You can instead use `n=1`
and rerun the cell to inspect one arrival at a time.

In [ ]:
_ = play(stream, n=len(observations) - stream.position)

## 4. Change allowed delay and explain
Set the second delay below, then run the cell. Each comparison starts with completely
fresh state and the same arrivals. Compare the late logs and retained minutes.

**Predict:** how will a longer delay affect finalization and retained state?

**Your prediction:** _Write here._

In [ ]:
ALLOWED_DELAY_SECONDS = 90  # Change this and rerun this cell.
short_run = replay(30)
changed_run = replay(ALLOWED_DELAY_SECONDS)
print("30-second run:", short_run.snapshot())
print("Changed-delay run:", changed_run.snapshot())

**Explain the tradeoff:** use one observation's route and the retained minutes as
evidence. Can the dashboard display provisional counts before finalization?
Why do active minutes remain after the last supplied observation?

**Your explanation:** _Write here._

**Exit:** Does a watermark prove that older observations cannot arrive? Why?

_Your answer:_

## Optional: see the real thing, not needed in class
The required activity is complete. Leave `RUN_SPARK = False` for normal Run all.
This opt-in illustration requires a runtime where PySpark and Java are already
available; nothing here installs them. Otherwise it prints a skip message.

The bus table is tiny and in memory. A test rate stream supplies its sequence IDs
in a micro-batch, and the query keeps its output in memory. `withWatermark` and
`window` perform the same event-time grouping using Structured Streaming.
These are implementation names for the watermark and window already defined.
The supplied batch does **not** reproduce the per-record replay's exact retirement
or late-log behavior. The query is stopped after a bounded wait.

[Optional implementation reference: Spark 4.0.1](https://spark.apache.org/docs/4.0.1/streaming/apis-on-dataframes-and-datasets.html#window-operations-on-event-time).

In [ ]:
# Default false: Run all needs only Python/pandas, and never installs anything.
RUN_SPARK = False
if not RUN_SPARK:
    print("Optional Spark demonstration skipped; required activity is complete.")
else:
    import importlib.util
    if importlib.util.find_spec("pyspark") is None:
        print("PySpark is not available. Skip this optional cell; do not install for class.")
    else:
        import tempfile
        from uuid import uuid4
        from pyspark.sql import SparkSession, functions as F
        spark = SparkSession.builder.master("local[2]").appName("bus-minute-demo").getOrCreate()
        spark.conf.set("spark.sql.session.timeZone", "UTC")
        spark.conf.set("spark.sql.shuffle.partitions", "2")
        # The bus observations live in a tiny in-memory table. A test rate stream
        # supplies matching sequence IDs, once, to make this a genuine streaming query.
        local_rows = [(i, r.observation_id, r.event_time.to_pydatetime().replace(tzinfo=None))
                      for i, r in observations.iterrows()]
        bus_table = spark.createDataFrame(local_rows, "seq long, observation_id string, event_time timestamp")
        tick = (spark.readStream.format("rate-micro-batch").option("rowsPerBatch", len(local_rows))
                .option("numPartitions", 1).load())
        bus_stream = (tick.where(F.col("value") < len(local_rows))
                      .join(bus_table, tick.value == bus_table.seq).select("observation_id", "event_time"))
        result = (bus_stream.withWatermark("event_time", "30 seconds")
                  .groupBy(F.window("event_time", "1 minute")).count())
        name = "bus_minutes_" + uuid4().hex
        query = None
        try:
            with tempfile.TemporaryDirectory() as checkpoint:
                query = (result.writeStream.format("memory").queryName(name).outputMode("update")
                         .option("checkpointLocation", checkpoint).trigger(availableNow=True).start())
                try:
                    query.awaitTermination(30)
                    spark.table(name).orderBy("window.start").show(truncate=False)
                finally:
                    query.stop()
        finally:
            spark.catalog.dropTempView(name)
            spark.stop()